# 02 — Silver: tipagem, deduplicação e o join AF ↔ PI

Transforma o texto cru do bronze em tabelas tipadas e resolve a ligação entre o AF e o Data Archive.

**Decisões desta camada**

- **Deduplicação por execução.** Se um dia teve mais de um run de metadados, vale o mais recente daquele `snapshot_date`. Health mantém todos os runs — é dado intradiário.
- **Join AF↔PI em duas chaves.** Primeiro `resolved_point_id` ↔ `point_id`, que sobrevive a rename da tag. Só depois `resolved_tag` ↔ `name`. `match_method` registra qual foi usada.
- **Divergência entre as chaves é sinal.** Se o ID resolve para uma tag de nome diferente do `resolved_tag`, alguém renomeou a tag e o AF ficou com o nome antigo no ConfigString.
- **Nada é descartado por ser suspeito.** O silver classifica; quem decide o que alertar é o gold.

> Timestamps vêm em ISO-8601 com offset explícito (`-03:00`). Aparecer `-02:00` em datas antigas é esperado: o Brasil teve horário de verão até 2019. O que quebraria o parser seria misturar `Z` com `±hh:mm`, o que não acontece.

In [ ]:
BRONZE = "bronze_"
SILVER = "silver_"

# Todo timestamp do PI sai neste formato. Ver docs/armadilhas.md.
TS = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

from pyspark.sql.utils import AnalysisException


def table_exists(name):
    try:
        spark.table(name)
        return True
    except AnalysisException:
        return False


def run(sql):
    spark.sql(sql)


def report(name):
    if table_exists(name):
        print(f"{name:34} {spark.table(name).count():>9,} linhas")
    else:
        print(f"{name:34} (não gerada)")

## Calendário de snapshots

Base das views de delta: qual snapshot veio antes e depois de qual, e quantas horas separaram os dois. Sem isso não há como exigir **dois snapshots consecutivos** antes de declarar que uma tag nasceu ou morreu.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {SILVER}snapshot_run AS
WITH runs AS (
    SELECT DISTINCT
        run_id,
        run_type,
        CAST(snapshot_date AS DATE)         AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')   AS snapshot_ts,
        status,
        CAST(run_duration_ms AS BIGINT)     AS run_duration_ms,
        module_version,
        pi_data_archive_version,
        af_server_version,
        CAST(row_count_total AS BIGINT)     AS row_count_total
    FROM {BRONZE}run_manifest
)
SELECT
    *,
    LAG(snapshot_date)  OVER (PARTITION BY run_type ORDER BY snapshot_ts) AS prev_snapshot_date,
    LAG(snapshot_ts)    OVER (PARTITION BY run_type ORDER BY snapshot_ts) AS prev_snapshot_ts,
    LEAD(snapshot_date) OVER (PARTITION BY run_type ORDER BY snapshot_ts) AS next_snapshot_date
FROM runs
""")

report(f"{SILVER}snapshot_run")

## PI Points

`dim_pipoint` guarda um registro por tag por dia. `fact_pipoint_health` preserva todos os runs do dia, porque é justamente a variação intradiária que revela interface parada.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {SILVER}dim_pipoint AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)         AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')   AS snapshot_ts,
        name,
        CAST(pointid AS INT)                AS point_id,
        pointtype                           AS point_type,
        ptclassname                         AS pt_class_name,
        descriptor,
        engunits                            AS eng_units,
        pointsource                         AS point_source,
        CAST(location1 AS INT) AS location1, CAST(location2 AS INT) AS location2,
        CAST(location3 AS INT) AS location3, CAST(location4 AS INT) AS location4,
        CAST(location5 AS INT) AS location5,
        instrumenttag                       AS instrument_tag,
        exdesc                              AS ex_desc,
        LOWER(compressing) IN ('true','1')  AS compressing,
        CAST(compdev AS DOUBLE)             AS comp_dev,
        CAST(excdev AS DOUBLE)              AS exc_dev,
        CAST(zero AS DOUBLE)                AS zero_value,
        CAST(span AS DOUBLE)                AS span_value,
        digitalset                          AS digital_set,
        TO_TIMESTAMP(creationdate, '{TS}')  AS creation_date,
        creator,
        TO_TIMESTAMP(changedate, '{TS}')    AS change_date,
        changer,
        LOWER(has_sensitive_hint) = 'true'  AS has_sensitive_hint,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY snapshot_date, pointid
            ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}pipoints
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

report(f"{SILVER}dim_pipoint")

In [ ]:
# Health é intradiário: todos os runs do dia são preservados.
run(f"""
CREATE OR REPLACE TABLE {SILVER}fact_pipoint_health AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)             AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')       AS snapshot_ts,
        name,
        CAST(pointid AS INT)                    AS point_id,
        pointtype                               AS point_type,
        pointsource                             AS point_source,
        snapshot_value,
        snapshot_status,
        TO_TIMESTAMP(snapshot_time, '{TS}')     AS value_time,
        LOWER(is_good) = 'true'                 AS is_good,
        CAST(age_hours AS DOUBLE)               AS age_hours,
        CAST(age_days AS DOUBLE)                AS age_days,
        LOWER(is_stale) = 'true'                AS is_stale,
        LOWER(is_pt_created) = 'true'           AS is_pt_created,
        LOWER(has_ever_received_data) = 'true'  AS has_ever_received_data,
        TO_TIMESTAMP(creationdate, '{TS}')      AS creation_date,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY _run_id, pointid ORDER BY _ingested_at DESC
        ) AS rn
    FROM {BRONZE}pipoint_health
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

report(f"{SILVER}fact_pipoint_health")

## AF: elementos e atributos

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {SILVER}dim_element AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)         AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')   AS snapshot_ts,
        element_path,
        element_name,
        parent_path,
        CAST(element_level AS INT)          AS element_level,
        template_name,
        categories,
        description,
        CAST(child_count AS INT)            AS child_count,
        CAST(attribute_count AS INT)        AS attribute_count,
        TO_TIMESTAMP(creation_date, '{TS}') AS creation_date,
        TO_TIMESTAMP(modify_date, '{TS}')   AS modify_date,
        guid                                AS element_guid,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY snapshot_date, guid ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}elements
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

# Atributos tipados, ainda sem o join: o join precisa de dim_pipoint pronto.
run(f"""
CREATE OR REPLACE TABLE {SILVER}stg_attribute AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)             AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')       AS snapshot_ts,
        element_path, element_name, element_template,
        attribute_path, attribute_name,
        CAST(attribute_level AS INT)            AS attribute_level,
        description, type AS attribute_type, uom, categories, template_name,
        data_reference, config_string, resolved_server, resolved_tag,
        CAST(resolved_point_id AS INT)          AS resolved_point_id,
        LOWER(is_dynamic_reference) = 'true'    AS is_dynamic_reference,
        parse_status,
        LOWER(is_configuration_item) = 'true'   AS is_configuration_item,
        LOWER(is_hidden) = 'true'               AS is_hidden,
        LOWER(is_excluded) = 'true'             AS is_excluded,
        guid                                    AS attribute_guid,
        element_guid,
        LOWER(has_sensitive_hint) = 'true'      AS has_sensitive_hint,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY snapshot_date, guid ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}attributes
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

report(f"{SILVER}dim_element")
report(f"{SILVER}stg_attribute")

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {SILVER}dim_attribute AS
WITH by_id AS (
    SELECT a.*, p.point_id AS m_id_point_id, p.name AS m_id_point_name
    FROM {SILVER}stg_attribute a
    LEFT JOIN {SILVER}dim_pipoint p
           ON p.snapshot_date = a.snapshot_date
          AND p.point_id      = a.resolved_point_id
),
by_name AS (
    SELECT b.*, p.point_id AS m_nm_point_id, p.name AS m_nm_point_name
    FROM by_id b
    LEFT JOIN {SILVER}dim_pipoint p
           ON p.snapshot_date = b.snapshot_date
          AND p.name          = b.resolved_tag
          AND b.m_id_point_id IS NULL
)
SELECT
    * EXCEPT (m_id_point_id, m_id_point_name, m_nm_point_id, m_nm_point_name),
    COALESCE(m_id_point_id, m_nm_point_id)     AS matched_point_id,
    COALESCE(m_id_point_name, m_nm_point_name) AS matched_point_name,
    CASE
        WHEN m_id_point_id IS NOT NULL THEN 'point_id'
        WHEN m_nm_point_id IS NOT NULL THEN 'name'
        ELSE NULL
    END AS match_method,
    CASE
        WHEN parse_status = 'not_pi_point'                      THEN 'not_pi_point'
        WHEN parse_status = 'empty'                             THEN 'no_config'
        WHEN parse_status = 'dynamic_tag'                       THEN 'dynamic'
        WHEN parse_status = 'unparsed'                          THEN 'unparsed'
        WHEN COALESCE(m_id_point_id, m_nm_point_id) IS NOT NULL THEN 'ok'
        ELSE 'broken'
    END AS dr_status,
    -- ID resolve para tag de nome diferente: renomearam a tag e o AF guardou o nome antigo.
    (m_id_point_id IS NOT NULL AND resolved_tag IS NOT NULL AND m_id_point_name <> resolved_tag)
        AS name_id_mismatch
FROM by_name
""")

report(f"{SILVER}dim_attribute")

print("\nDistribuição de dr_status no snapshot mais recente:")
spark.sql(f"""
SELECT dr_status, COUNT(*) AS qtd
FROM {SILVER}dim_attribute
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM {SILVER}dim_attribute)
GROUP BY dr_status ORDER BY qtd DESC
""").show(truncate=False)

### O join AF ↔ PI

Duas chaves, nesta ordem de confiança:

| Chave | Por quê |
|---|---|
| `resolved_point_id` ↔ `point_id` | sobrevive a rename da tag |
| `resolved_tag` ↔ `name` | funciona quando o DR não guarda o ID |

A comparação é sempre **dentro do mesmo `snapshot_date`**: cruzar o AF de hoje com o PI de ontem inventaria DR quebrado que não existe.

`dr_status` classifica cada atributo, e é dele que saem as views de tag órfã e DR quebrado. Repare que `unparsed` é caso à parte — indica bug no parser de `ConfigString`, não configuração errada no AF.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {SILVER}dim_template AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)             AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')       AS snapshot_ts,
        row_type, template_name, instance_type, base_template,
        categories, description,
        attribute_template_name, attribute_template_path,
        CAST(attribute_level AS INT)            AS attribute_level,
        type AS attribute_type, uom,
        data_reference, config_string, resolved_tag,
        LOWER(is_dynamic_reference) = 'true'    AS is_dynamic_reference,
        parse_status,
        LOWER(is_configuration_item) = 'true'   AS is_configuration_item,
        guid,
        LOWER(has_sensitive_hint) = 'true'      AS has_sensitive_hint,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY snapshot_date, guid, row_type
            ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}templates
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

# Event frames são imutáveis depois de fechados, e a janela de 30 dias os recoleta todo dia.
# A deduplicação é por GUID, não por snapshot, para não multiplicar cada EF por 30.
run(f"""
CREATE OR REPLACE TABLE {SILVER}fact_eventframe AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)             AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')       AS snapshot_ts,
        event_frame_name, template_name,
        TO_TIMESTAMP(start_time, '{TS}')        AS start_time,
        TO_TIMESTAMP(end_time, '{TS}')          AS end_time,
        CAST(duration_minutes AS DOUBLE)        AS duration_minutes,
        LOWER(is_open) = 'true'                 AS is_open,
        primary_element, primary_element_path,
        CAST(referenced_element_count AS INT)   AS referenced_element_count,
        severity, categories, description,
        LOWER(are_values_captured) = 'true'     AS are_values_captured,
        parent_guid, guid AS event_frame_guid,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY guid ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}eventframes
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

run(f"""
CREATE OR REPLACE TABLE {SILVER}dim_analysis AS
WITH ranked AS (
    SELECT
        CAST(snapshot_date AS DATE)             AS snapshot_date,
        TO_TIMESTAMP(snapshot_ts, '{TS}')       AS snapshot_ts,
        analysis_name, analysis_path, target_path, target_name, template_name,
        analysis_rule, time_rule, categories, description,
        configured_status,
        LOWER(is_configured) = 'true'           AS is_configured,
        LOWER(auto_created) = 'true'            AS auto_created,
        LOWER(has_notification) = 'true'        AS has_notification,
        priority,
        runtime_status,
        last_evaluation_status,
        TO_TIMESTAMP(last_trigger_time, '{TS}') AS last_trigger_time,
        CAST(error_count AS BIGINT)             AS error_count,
        CAST(success_count AS BIGINT)           AS success_count,
        CAST(skip_count AS BIGINT)              AS skip_count,
        CAST(last_elapsed_ms AS DOUBLE)         AS last_elapsed_ms,
        status_source,
        TO_TIMESTAMP(creation_date, '{TS}')     AS creation_date,
        TO_TIMESTAMP(modify_date, '{TS}')       AS modify_date,
        guid AS analysis_guid,
        _run_id,
        ROW_NUMBER() OVER (
            PARTITION BY snapshot_date, guid ORDER BY snapshot_ts DESC, _ingested_at DESC
        ) AS rn
    FROM {BRONZE}analyses
)
SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
""")

for t in ["dim_template", "fact_eventframe", "dim_analysis"]:
    report(SILVER + t)

## Templates, event frames e analyses

In [ ]:
# Tabelas de apoio: cópia deduplicada, sem regra de negócio.
for src, key in [("digitalsets", "set_name, state_value"),
                 ("enumsets", "set_guid, value"),
                 ("uom", "guid"),
                 ("tables", "guid")]:
    if not table_exists(BRONZE + src):
        continue
    run(f"""
    CREATE OR REPLACE TABLE {SILVER}dim_{src} AS
    WITH ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY snapshot_date, {key}
                   ORDER BY snapshot_ts DESC, _ingested_at DESC
               ) AS rn
        FROM {BRONZE}{src}
    )
    SELECT * EXCEPT (rn) FROM ranked WHERE rn = 1
    """)
    report(f"{SILVER}dim_{src}")

In [ ]:
# Sanidade: a taxa de resolução do parser é a falha mais provável do projeto, e é silenciosa.
print("Qualidade do join AF -> PI no snapshot mais recente:")
spark.sql(f"""
SELECT
    COUNT(*)                                                          AS atributos,
    SUM(CASE WHEN dr_status = 'ok' THEN 1 ELSE 0 END)                 AS ligados,
    SUM(CASE WHEN dr_status = 'broken' THEN 1 ELSE 0 END)             AS dr_quebrado,
    SUM(CASE WHEN dr_status = 'unparsed' THEN 1 ELSE 0 END)           AS parser_falhou,
    SUM(CASE WHEN name_id_mismatch THEN 1 ELSE 0 END)                 AS renomeadas,
    SUM(CASE WHEN match_method = 'point_id' THEN 1 ELSE 0 END)        AS casadas_por_id,
    SUM(CASE WHEN match_method = 'name' THEN 1 ELSE 0 END)            AS casadas_por_nome
FROM {SILVER}dim_attribute
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM {SILVER}dim_attribute)
""").show(truncate=False)

print("Se 'parser_falhou' for maior que zero, investigue o parser antes de acusar o AF.")